# Modul 2: Projektowanie nagrod RL — reward shaping dla G1

W tym notebooku zbadamy **reward shaping** — kluczowy element treningu RL dla lokomotion humanoidow.

Nagrody w RL pelnia role "jezyka", w ktorym komunikujemy robotowi czego od niego oczekujemy.
Zle zaprojektowane nagrody prowadza do **reward hackingu** — robot znajduje sposoby na maksymalizacje
nagrody, ktore nie odpowiadaja naszym intencjom (np. drga w miejscu zbierajac alive bonus).

**Plan notebooka:**
1. Zaladowanie modelu G1 w Mujoco
2. Zrozumienie observation space (45D)
3. Implementacja komponentow nagrody
4. Symulacja z kontrolerem PD i wizualizacja nagrod
5. Eksperyment: porownanie roznych konfiguracji wag
6. Wizualizacja zachowania robota
7. Export do symulatora przegladarkowego

In [ ]:
# Instalacja zaleznosci
!pip install -q mujoco mujoco_menagerie numpy matplotlib

In [ ]:
import mujoco
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Zaladuj model G1 z mujoco_menagerie
import mujoco_menagerie
MODEL_PATH = str(Path(mujoco_menagerie.__path__[0]) / "unitree_g1" / "scene.xml")

model = mujoco.MjModel.from_xml_path(MODEL_PATH)
data = mujoco.MjData(model)

# Reset do pozycji stojacej (keyframe)
kf_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_KEY, 0)
data.qpos[:] = model.keyframe(kf_name).qpos
data.qvel[:] = 0
mujoco.mj_forward(model, data)

standing_qpos = data.qpos.copy()
target_positions = data.qpos[7:].copy()  # domyslne pozycje stawow

print(f"Model: {MODEL_PATH}")
print(f"Liczba stawow (njnt): {model.njnt}")
print(f"Liczba aktuatorow (nu): {model.nu}")
print(f"Wymiar qpos: {model.nq}, qvel: {model.nv}")
print(f"Wysokosc stojacego robota: {data.qpos[2]:.3f} m")
print(f"Keyframe: '{kf_name}'")

## Obserwacje robota (observation space)

Policy RL dla G1 otrzymuje **45-wymiarowy** wektor obserwacji (actor):

| Obserwacja | Dim | Opis |
|---|---|---|
| `base_ang_vel` | 3 | Predkosc katowa bazy w ramce ciala, skala 0.25 |
| `projected_gravity` | 3 | Wektor grawitacji w ramce ciala (dla upright: [0, 0, -1]) |
| `velocity_commands` | 3 | Komendy predkosci: vx, vy, wz |
| `joint_pos_rel` | 12 | Pozycje stawow wzgledem domyslnych |
| `joint_vel_rel` | 12 | Predkosci stawow, skala 0.05 |
| `last_action` | 12 | Poprzednia akcja |

**Critic** dodatkowo otrzymuje `base_lin_vel` (3D) — informacja uprzywilejowana niedostepna na realnym robocie.

Akcje to **delta joint positions** — siec neuronowa generuje wartosci w [-1, 1], ktore sa skalowane:
```
target_q = default_q + action * action_scale  (action_scale = 0.25)
torque = kp * (target_q - current_q) + kd * (0 - current_dq)
```

In [ ]:
def quat_rotate_inverse(q, v):
    """Obroc wektor v przez odwrotnosc kwaternionu q (format wxyz)."""
    w, x, y, z = q
    q_conj = np.array([w, -x, -y, -z])
    v_quat = np.array([0, v[0], v[1], v[2]])

    def quat_mul(a, b):
        return np.array([
            a[0]*b[0] - a[1]*b[1] - a[2]*b[2] - a[3]*b[3],
            a[0]*b[1] + a[1]*b[0] + a[2]*b[3] - a[3]*b[2],
            a[0]*b[2] - a[1]*b[3] + a[2]*b[0] + a[3]*b[1],
            a[0]*b[3] + a[1]*b[2] - a[2]*b[1] + a[3]*b[0],
        ])

    result = quat_mul(quat_mul(q_conj, v_quat), q)
    return result[1:4]


def compute_observations(data, model, last_action, commands):
    """Oblicz 45D wektor obserwacji z MjData (identycznie jak unitree_rl_lab).

    Skladniki:
    - ang_vel_body (3): predkosc katowa bazy w ramce ciala
    - projected_gravity (3): wektor grawitacji w ramce ciala
    - commands (3): komendy predkosci [vx, vy, wz]
    - joint_pos_rel (12): pozycje stawow wzgledem domyslnych
    - joint_vel (12): predkosci stawow
    - last_action (12): poprzednia akcja
    """
    # Predkosc katowa bazy w ramce ciala
    quat = data.qpos[3:7]  # wxyz z Mujoco
    ang_vel_world = data.qvel[3:6]
    ang_vel_body = quat_rotate_inverse(quat, ang_vel_world)

    # Projected gravity — jak robot "czuje" grawitacje
    gravity_world = np.array([0.0, 0.0, -1.0])
    projected_gravity = quat_rotate_inverse(quat, gravity_world)

    # Pozycje stawow wzgledem domyslnych (12 kontrolowanych nog)
    # Kontrolowane stawy — 12 stawow nog
    controlled_joint_names = [
        "left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint",
        "left_knee_joint", "left_ankle_pitch_joint", "left_ankle_roll_joint",
        "right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint",
        "right_knee_joint", "right_ankle_pitch_joint", "right_ankle_roll_joint",
    ]
    joint_indices = []
    for name in controlled_joint_names:
        jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)
        if jid >= 0:
            joint_indices.append(jid)

    default_pos = np.zeros(len(joint_indices))
    joint_pos = np.array([data.qpos[model.jnt_qposadr[ji]] for ji in joint_indices])
    joint_pos_rel = joint_pos - default_pos

    joint_vel = np.array([data.qvel[model.jnt_dofadr[ji]] for ji in joint_indices])

    # Skale obserwacji (jak w unitree_rl_lab)
    obs = np.concatenate([
        ang_vel_body * 0.25,          # 3  — ang_vel_scale
        projected_gravity,            # 3
        np.array(commands),           # 3  — [vx, vy, wz]
        joint_pos_rel * 1.0,          # 12 — dof_pos_scale
        joint_vel * 0.05,             # 12 — dof_vel_scale
        np.array(last_action),        # 12
    ])

    return obs, {
        'ang_vel_body': ang_vel_body,
        'projected_gravity': projected_gravity,
        'joint_pos': joint_pos,
        'joint_vel': joint_vel,
        'joint_indices': joint_indices,
    }


# Test: obserwacja stojacego robota
data.qpos[:] = standing_qpos
data.qvel[:] = 0
mujoco.mj_forward(model, data)

obs, obs_info = compute_observations(data, model, np.zeros(12), [0, 0, 0])
print(f"Wymiar obserwacji: {obs.shape[0]}D")
print(f"\nProjected gravity (stojacy): {obs_info['projected_gravity']}")
print(f"  (oczekiwane: [0, 0, -1] dla robota stojacego pionowo)")
print(f"\nAng vel body: {obs_info['ang_vel_body']}")
print(f"  (oczekiwane: [0, 0, 0] dla nieruchomego robota)")

## Komponenty nagrody

Kazda nagroda w RL dla lokomotion ma okreslona role:

| Nagroda | Typ | Cel |
|---|---|---|
| `alive_reward` | + | Bonus za kazdy krok (zacheta do dlugich epizodow) |
| `height_reward` | - | Kara za odchylenie od docelowej wysokosci (0.78m) |
| `orientation_penalty` | - | Kara za odchylenie tulowia od pionu |
| `tracking_lin_vel` | + | Nagroda za sledzenie zadanej predkosci |
| `angular_velocity_penalty` | - | Kara za nadmierna rotacje |
| `energy_penalty` | - | Kara za zuzycie energii (tau x dq) |
| `joint_limit_penalty` | - | Kara za zblizanie sie do limitow stawow |
| `action_rate_penalty` | - | Kara za gwaltowne zmiany akcji |

Kluczowe: **rownowaga miedzy nagrodami** — zbyt wysoka waga jednej kary moze zablokowac uczenie.

In [ ]:
# ─── Komponenty nagrody (jak w unitree_rl_lab / IsaacLab) ───

def alive_reward(data, min_height=0.3):
    """Bonus za zycie: +1 jesli robot nie upadl (wysokosc > min_height).
    Zacheca do dlugich epizodow."""
    height = data.qpos[2]
    return 1.0 if height > min_height else 0.0


def height_reward(data, target=0.78):
    """Kara za odchylenie od docelowej wysokosci.
    Robot G1 stojacy prosto ma ~0.79m."""
    height = data.qpos[2]
    return -((height - target) ** 2)


def orientation_penalty(data):
    """Kara za odchylenie od pionu (projected gravity).
    Dla robota stojacego prosto: projected_gravity = [0, 0, -1]."""
    quat = data.qpos[3:7]
    w, x, y, z = quat
    # Trzecia kolumna macierzy rotacji (os z ciala w swiecie)
    bz_x = 2 * (x * z + w * y)
    bz_y = 2 * (y * z - w * x)
    return -(bz_x**2 + bz_y**2)


def tracking_lin_vel_reward(data, cmd_vx=0.5, cmd_vy=0.0):
    """Nagroda za sledzenie zadanej predkosci (exp kernel).
    1.0 = idealne sledzenie, 0.0 = duzy blad."""
    vx = data.qvel[0]
    vy = data.qvel[1]
    error = (vx - cmd_vx)**2 + (vy - cmd_vy)**2
    return np.exp(-error / 0.25)


def angular_velocity_penalty(data):
    """Kara za nadmierna predkosc katowa bazy.
    Stabilny robot nie powinien sie szybko krecic."""
    ang_vel = data.qvel[3:6]
    return -np.sum(ang_vel**2)


def energy_penalty(data):
    """Kara za zuzycie energii: sum(|torque * velocity|).
    Zacheca do efektywnego ruchu."""
    torques = data.actuator_force
    velocities = data.qvel[6:]  # pomijamy free joint
    return -np.sum(np.abs(torques * velocities[:len(torques)]))


def joint_limit_penalty(data, model):
    """Kara za zblizanie sie do limitow stawow.
    Margines 10% zakresu — kara rosnie kwadratowo."""
    penalty = 0.0
    for i in range(model.njnt):
        if model.jnt_type[i] != 3:  # pomijamy nie-hinge
            continue
        q = data.qpos[7 + i - 1]
        lo, hi = model.jnt_range[i]
        margin = 0.1
        range_size = hi - lo
        if range_size <= 0:
            continue
        if q < lo + margin * range_size:
            penalty += ((lo + margin * range_size - q) / (margin * range_size)) ** 2
        elif q > hi - margin * range_size:
            penalty += ((q - (hi - margin * range_size)) / (margin * range_size)) ** 2
    return -penalty


def action_rate_penalty(prev_action, curr_action):
    """Kara za gwaltowne zmiany akcji (gladkosc ruchu)."""
    return -np.sum((curr_action - prev_action) ** 2)


# ─── Walidacja: test na stojacym robocie ───
data.qpos[:] = standing_qpos
data.qvel[:] = 0
mujoco.mj_forward(model, data)

print("=== Nagrody dla stojacego robota (idealny stan) ===")
print(f"  alive_reward:             {alive_reward(data):+.4f}  (1.0 = zyje)")
print(f"  height_reward (t=0.78):   {height_reward(data, 0.78):+.6f}  (0 = idealna wysokosc)")
print(f"  orientation_penalty:      {orientation_penalty(data):+.6f}  (0 = pionowo)")
print(f"  tracking_lin_vel (cmd=0): {tracking_lin_vel_reward(data, 0, 0):.4f}  (1 = idealne)")
print(f"  angular_vel_penalty:      {angular_velocity_penalty(data):+.6f}  (0 = brak rotacji)")
print(f"  energy_penalty:           {energy_penalty(data):+.4f}  (0 = brak ruchu)")
print(f"  joint_limit_penalty:      {joint_limit_penalty(data, model):+.4f}  (0 = daleko od limitow)")

# Test: przechylony robot
print("\n=== Nagrody dla przechylonego robota (30 stopni do przodu) ===")
data.qpos[:] = standing_qpos
angle = np.radians(30)
data.qpos[3] = np.cos(angle / 2)
data.qpos[4] = 0
data.qpos[5] = np.sin(angle / 2)
data.qpos[6] = 0
mujoco.mj_forward(model, data)
print(f"  orientation_penalty:      {orientation_penalty(data):+.6f}  (powinno byc ujemne)")
print(f"  height_reward (t=0.78):   {height_reward(data, 0.78):+.6f}")

In [ ]:
# ─── Symulacja z prostym kontrolerem PD + obliczanie nagrod ───

def run_simulation_with_rewards(model, duration=3.0, weights=None, cmd_vx=0.0):
    """Symuluj robota z kontrolerem PD i oblicz nagrody w kazdym kroku.

    Args:
        model: MjModel
        duration: czas symulacji w sekundach
        weights: dict z wagami nagrod
        cmd_vx: komenda predkosci do przodu

    Returns:
        dict z historią nagrod i stanem robota
    """
    if weights is None:
        weights = {
            'alive': 2.0,
            'height': -15.0,
            'orientation': -5.0,
            'tracking_vel': 1.5,
            'angular_vel': -0.05,
            'energy': -0.001,
            'joint_limits': -10.0,
            'action_rate': -0.01,
        }

    d = mujoco.MjData(model)
    kf_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_KEY, 0)
    d.qpos[:] = model.keyframe(kf_name).qpos
    d.qvel[:] = 0
    mujoco.mj_forward(model, d)

    dt = model.opt.timestep
    n_steps = int(duration / dt)
    policy_decimation = 10  # policy co 10 krokow fizyki

    # PD gains
    kp = 50.0
    kd = 2.0
    default_q = d.qpos[7:].copy()

    # Historia
    history = {
        'time': [],
        'height': [],
        'alive': [],
        'height_rew': [],
        'orientation': [],
        'tracking_vel': [],
        'angular_vel': [],
        'energy': [],
        'joint_limits': [],
        'action_rate': [],
        'total': [],
        'vx': [],
    }

    prev_action = np.zeros(model.nu)

    for step in range(n_steps):
        # Prosty kontroler PD: utrzymuj domyslna pozycje + maly ruch
        # Symulujemy "niedoskonala" policy — lekkie odchylenia
        t = step * dt
        action_noise = 0.02 * np.sin(2 * np.pi * t * np.array(
            [1.0 + 0.1*i for i in range(model.nu)]
        ))
        target_q = default_q[:model.nu] + action_noise

        curr_q = d.qpos[7:7+model.nu]
        curr_dq = d.qvel[6:6+model.nu]
        d.ctrl[:] = kp * (target_q - curr_q) + kd * (0 - curr_dq)

        mujoco.mj_step(model, d)

        # Oblicz nagrody co policy_decimation krokow
        if step % policy_decimation == 0:
            r_alive = alive_reward(d)
            r_height = height_reward(d, 0.78)
            r_orient = orientation_penalty(d)
            r_track = tracking_lin_vel_reward(d, cmd_vx, 0)
            r_angvel = angular_velocity_penalty(d)
            r_energy = energy_penalty(d)
            r_jlimit = joint_limit_penalty(d, model)
            r_arate = action_rate_penalty(prev_action, action_noise)

            total = (
                weights['alive'] * r_alive +
                weights['height'] * r_height +
                weights['orientation'] * r_orient +
                weights['tracking_vel'] * r_track +
                weights['angular_vel'] * r_angvel +
                weights['energy'] * r_energy +
                weights['joint_limits'] * r_jlimit +
                weights['action_rate'] * r_arate
            )

            history['time'].append(t)
            history['height'].append(d.qpos[2])
            history['alive'].append(weights['alive'] * r_alive)
            history['height_rew'].append(weights['height'] * r_height)
            history['orientation'].append(weights['orientation'] * r_orient)
            history['tracking_vel'].append(weights['tracking_vel'] * r_track)
            history['angular_vel'].append(weights['angular_vel'] * r_angvel)
            history['energy'].append(weights['energy'] * r_energy)
            history['joint_limits'].append(weights['joint_limits'] * r_jlimit)
            history['action_rate'].append(weights['action_rate'] * r_arate)
            history['total'].append(total)
            history['vx'].append(d.qvel[0])

            prev_action = action_noise.copy()

    # Konwertuj na numpy
    for key in history:
        history[key] = np.array(history[key])

    return history


# Uruchom symulacje z domyslnymi wagami
history = run_simulation_with_rewards(model, duration=3.0)

# Wykres wszystkich komponentow nagrody
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Panel 1: Nagrody pozytywne
axes[0].plot(history['time'], history['alive'], label='alive', color='green', linewidth=2)
axes[0].plot(history['time'], history['tracking_vel'], label='tracking_vel', color='blue', linewidth=2)
axes[0].set_ylabel('Nagroda (wazona)')
axes[0].set_title('Komponenty nagrody — pozytywne')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Panel 2: Kary
axes[1].plot(history['time'], history['height_rew'], label='height', color='red')
axes[1].plot(history['time'], history['orientation'], label='orientation', color='orange')
axes[1].plot(history['time'], history['angular_vel'], label='angular_vel', color='purple')
axes[1].plot(history['time'], history['energy'], label='energy', color='brown')
axes[1].plot(history['time'], history['joint_limits'], label='joint_limits', color='pink')
axes[1].plot(history['time'], history['action_rate'], label='action_rate', color='gray')
axes[1].set_ylabel('Kara (wazona)')
axes[1].set_title('Komponenty nagrody — kary')
axes[1].legend(loc='lower right', fontsize=8)
axes[1].grid(True, alpha=0.3)

# Panel 3: Nagroda calkowita + wysokosc
ax3 = axes[2]
ax3.plot(history['time'], history['total'], label='total reward', color='black', linewidth=2)
ax3.set_ylabel('Nagroda calkowita')
ax3.set_xlabel('Czas [s]')
ax3.set_title('Nagroda calkowita')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

ax3b = ax3.twinx()
ax3b.plot(history['time'], history['height'], label='wysokosc [m]', color='cyan', linestyle='--', alpha=0.7)
ax3b.set_ylabel('Wysokosc [m]')
ax3b.legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f"Srednia nagroda calkowita: {history['total'].mean():.4f}")
print(f"Koncowa wysokosc robota: {history['height'][-1]:.3f} m")

## Eksperyment: zmiana wag nagrod

Porownamy trzy konfiguracje wag:

1. **Domyslna** — zbalansowane wagi
2. **Niska energia** — wysoka kara za zuzycie energii (robot powinien oszczedzac energie)
3. **Wysoka stabilnosc** — wysoka kara za orientacje i wysokosc (robot priorytetyzuje stabilnosc)

To jest uproszczona wersja iteracyjnego procesu projektowania nagrod w prawdziwym treningu RL.
W IsaacLab zmiana wag wymaga ponownego treningu (tysiace iteracji), tutaj widzimy efekt od razu.

In [ ]:
# ─── Porownanie trzech konfiguracji wag ───

weight_configs = {
    'Domyslna': {
        'alive': 2.0, 'height': -15.0, 'orientation': -5.0,
        'tracking_vel': 1.5, 'angular_vel': -0.05,
        'energy': -0.001, 'joint_limits': -10.0, 'action_rate': -0.01,
    },
    'Niska energia': {
        'alive': 2.0, 'height': -15.0, 'orientation': -5.0,
        'tracking_vel': 1.5, 'angular_vel': -0.05,
        'energy': -0.1,       # 100x wieksza kara za energie!
        'joint_limits': -10.0, 'action_rate': -0.01,
    },
    'Wysoka stabilnosc': {
        'alive': 5.0,           # wyzszy bonus za zycie
        'height': -50.0,        # 3x wieksza kara za wysokosc
        'orientation': -20.0,   # 4x wieksza kara za orientacje
        'tracking_vel': 0.5,    # nizszy priorytet sledzenia predkosci
        'angular_vel': -0.5,    # 10x wieksza kara za rotacje
        'energy': -0.001, 'joint_limits': -10.0, 'action_rate': -0.01,
    },
}

results = {}
for name, weights in weight_configs.items():
    results[name] = run_simulation_with_rewards(model, duration=3.0, weights=weights)
    print(f"{name:20s} — srednia nagroda: {results[name]['total'].mean():+.4f}, "
          f"koncowa wysokosc: {results[name]['height'][-1]:.3f} m")

# Wykres porownawczy
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

colors = {'Domyslna': '#2196F3', 'Niska energia': '#4CAF50', 'Wysoka stabilnosc': '#FF5722'}

# Nagroda calkowita
ax = axes[0, 0]
for name, hist in results.items():
    ax.plot(hist['time'], hist['total'], label=name, color=colors[name], linewidth=2)
ax.set_title('Nagroda calkowita')
ax.set_ylabel('Total reward')
ax.legend()
ax.grid(True, alpha=0.3)

# Wysokosc
ax = axes[0, 1]
for name, hist in results.items():
    ax.plot(hist['time'], hist['height'], label=name, color=colors[name], linewidth=2)
ax.axhline(y=0.78, color='gray', linestyle='--', alpha=0.5, label='target (0.78m)')
ax.set_title('Wysokosc robota')
ax.set_ylabel('Wysokosc [m]')
ax.legend()
ax.grid(True, alpha=0.3)

# Kara za energie
ax = axes[1, 0]
for name, hist in results.items():
    ax.plot(hist['time'], hist['energy'], label=name, color=colors[name], linewidth=2)
ax.set_title('Kara za energie (wazona)')
ax.set_ylabel('Energy penalty')
ax.set_xlabel('Czas [s]')
ax.legend()
ax.grid(True, alpha=0.3)

# Kara za orientacje
ax = axes[1, 1]
for name, hist in results.items():
    ax.plot(hist['time'], hist['orientation'], label=name, color=colors[name], linewidth=2)
ax.set_title('Kara za orientacje (wazona)')
ax.set_ylabel('Orientation penalty')
ax.set_xlabel('Czas [s]')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Porownanie konfiguracji wag nagrod', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nWnioski:")
print("- 'Niska energia': robot minimalizuje ruch kosztem sledzenia predkosci")
print("- 'Wysoka stabilnosc': robot priorytetyzuje utrzymanie wysokosci i orientacji")
print("- Kluczowe: balans wag to iteracyjny proces — w treningu RL wymaga wielu prob")

## Wizualizacja — robot z roznymi nagrodami

Renderujemy klatki z symulacji dla kazdej konfiguracji wag.
W prawdziwym treningu RL roznice bylyby bardziej widoczne — tutaj uzywamy prostego kontrolera PD,
ale zasada jest ta sama: **wagi nagrod ksztaltuja zachowanie robota**.

In [ ]:
# ─── Renderowanie klatkowe ───

def render_simulation_frames(model, duration=2.0, n_frames=6, weights=None, cmd_vx=0.0):
    """Symuluj i renderuj klatki."""
    d = mujoco.MjData(model)
    kf_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_KEY, 0)
    d.qpos[:] = model.keyframe(kf_name).qpos
    d.qvel[:] = 0
    mujoco.mj_forward(model, d)

    # Renderer
    renderer = mujoco.Renderer(model, height=320, width=480)

    dt = model.opt.timestep
    n_steps = int(duration / dt)
    frame_interval = max(1, n_steps // n_frames)

    kp, kd = 50.0, 2.0
    default_q = d.qpos[7:].copy()

    frames = []
    frame_times = []

    for step in range(n_steps):
        t = step * dt
        action_noise = 0.02 * np.sin(2 * np.pi * t * np.array(
            [1.0 + 0.1*i for i in range(model.nu)]
        ))
        target_q = default_q[:model.nu] + action_noise
        curr_q = d.qpos[7:7+model.nu]
        curr_dq = d.qvel[6:6+model.nu]
        d.ctrl[:] = kp * (target_q - curr_q) + kd * (0 - curr_dq)

        mujoco.mj_step(model, d)

        if step % frame_interval == 0 and len(frames) < n_frames:
            renderer.update_scene(d)
            frame = renderer.render()
            frames.append(frame.copy())
            frame_times.append(t)

    renderer.close()
    return frames, frame_times


# Renderuj dla kazdej konfiguracji
fig, axes = plt.subplots(3, 6, figsize=(18, 9))

for row, (name, weights) in enumerate(weight_configs.items()):
    frames, times = render_simulation_frames(model, duration=2.0, n_frames=6, weights=weights)
    for col, (frame, t) in enumerate(zip(frames, times)):
        axes[row, col].imshow(frame)
        axes[row, col].set_title(f't={t:.2f}s', fontsize=9)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name, fontsize=12, fontweight='bold', rotation=0,
                            labelpad=80, va='center')

plt.suptitle('Zachowanie robota G1 — rozne konfiguracje wag nagrod',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Uwaga: z prostym kontrolerem PD roznice sa subtelne.")
print("W prawdziwym treningu RL (tysiace iteracji PPO) roznice sa dramatyczne.")

## Export do symulatora przegladarkowego

Po wytrenowaniu policy w IsaacLab, mozesz wyeksportowac trajektorie do formatu JSON
i odtworzyc ja w symulatorze przegladarkowym kursu.

Skrypt `export_trajectory.py` wykonuje **sim2sim** — uruchamia policy ONNX w Mujoco
i zapisuje kolejne klatki (qpos) do pliku JSON:

```bash
# Export trajektorii z wytrenowanej policy
python exercises/module2/export_trajectory.py \
    --policy policy.onnx \
    --config deploy.yaml \
    --duration 5.0 \
    --vx 0.5

# Wynik: trajectory.json
# Upload do symulatora przegladarkowego → tab 'Trajektoria' → Upload
```

**Format trajektorii:**
```json
{
  "name": "Policy: velocity_walk",
  "dt": 0.02,
  "nq": 37,
  "controlled_joints": ["left_hip_pitch_joint", ...],
  "commands": [0.5, 0.0, 0.0],
  "frames": [[qpos_t0], [qpos_t1], ...]
}
```

Pipeline: **IsaacLab (train)** → **ONNX (export)** → **Mujoco (sim2sim)** → **JSON (browser)**